# SEC/EDGAR — extração dirigida pelo painel da Compustat

Esta versão substitui a varredura por ano pela lista de empresas-ano definida pela professora:
10.163 empresas, exercícios de 2017 em diante, 66.680 observações.

**O que muda em relação ao notebook anterior.** Antes o programa percorria todos os relatórios
anuais protocolados na SEC, ano a ano. Agora ele parte do painel: para cada empresa e cada
exercício, localiza o relatório correspondente e extrai apenas esse. O motor de recorte — a
pontuação por sinais contábeis, as âncoras e a busca em anexos — permanece idêntico ao que foi
validado nos pilotos.

**Como o relatório é identificado.** A SEC informa, para cada protocolo, a data de encerramento
do período a que ele se refere. É por esse campo que o casamento é feito, e não pela data do
protocolo, o que resolve os casos de empresa que mudou de exercício social. Quando não há
correspondência exata, o programa aceita uma diferença de poucos dias, e só então recorre à
janela de datas de protocolo.

**Ordem de trabalho.** Seções 1 a 3 preparam o ambiente. A seção 4 monta a tabela de protocolos
consultando a SEC uma vez por empresa, e grava o resultado — é a etapa que leva cerca de vinte
minutos e não precisa ser repetida. A seção 5 extrai. A seção 6 confere e a 7 compacta.

**Organização da saída.** Os PDFs ficam em `pdf/<ano fiscal>/<formulário>/`, seguindo o ano
fiscal do painel, e não o ano do protocolo — assim cada arquivo cai no ano em que a professora
vai usá-lo.

## 1. Configuração

In [ ]:
SEU_NOME  = "Helena Ribeiro"
SEU_EMAIL = "helenafariasr@gmail.com"

USAR_DRIVE = True
PASTA      = "/content/drive/MyDrive/SEC_PAINEL"

# Arquivo da Compustat, no Drive. Faça o upload dele para a pasta acima antes de rodar.
ARQUIVO_PAINEL = "/content/drive/MyDrive/SEC_PAINEL/Compustat Companies Filtered1.csv"

FORMULARIOS = ["10-K", "20-F", "40-F"]
TOLERANCIA_DIAS = 15      # diferença aceita entre o fim do exercício e o período do relatório

SALVAR_HTML_DA_SECAO = False

REQ_POR_SEGUNDO = 8
N_THREADS       = 4
MAX_TENTATIVAS  = 4

In [ ]:
import os

if USAR_DRIVE:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive", force_remount=True)
        print("Drive montado.")
    except Exception as e:
        print("Não foi possível montar o Drive:", type(e).__name__, e)
        USAR_DRIVE = False
        PASTA = PASTA.replace("/content/drive/MyDrive", "/content")

for sub in ["", "indice", "pdf", "html_secao", "log"]:
    os.makedirs(os.path.join(PASTA, sub), exist_ok=True)
print("Pasta de trabalho:", PASTA)

!apt-get -qq update > /dev/null && apt-get -qq install -y wkhtmltopdf poppler-utils > /dev/null
!wkhtmltopdf --version | head -1 && pdftotext -v 2>&1 | head -1

Mounted at /content/drive
Drive montado.
Pasta de trabalho: /content/drive/MyDrive/SEC_PAINEL
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
wkhtmltopdf 0.12.6
pdftotext version 22.02.0


In [ ]:
import io, os, re, json, time, random, threading, subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin

import requests
import pandas as pd

assert "@" in SEU_EMAIL and SEU_NOME.strip(), "Preencha SEU_NOME e SEU_EMAIL."

HEADERS = {"User-Agent": f"{SEU_NOME} {SEU_EMAIL}", "Accept-Encoding": "gzip, deflate"}

_lock, _ultimo = threading.Lock(), [0.0]

def _esperar_vez():
    with _lock:
        agora = time.time()
        espera = _ultimo[0] + 1.0 / REQ_POR_SEGUNDO - agora
        if espera > 0:
            time.sleep(espera)
            agora = time.time()
        _ultimo[0] = agora

_local = threading.local()

def _sessao():
    if not hasattr(_local, "s"):
        s = requests.Session(); s.headers.update(HEADERS); _local.s = s
    return _local.s

def baixar(url, binario=False):
    """GET com limite de taxa, repetição e espera crescente. None se falhar."""
    for i in range(MAX_TENTATIVAS):
        _esperar_vez()
        try:
            r = _sessao().get(url, timeout=90)
            if r.status_code == 200:
                return r.content if binario else r.text
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep((2 ** i) + random.random())
    return None

print("Identificação enviada à SEC:", HEADERS["User-Agent"])

Identificação enviada à SEC: Helena Ribeiro helenafariasr@gmail.com


## 2. Motor de recorte

Idêntico ao da versão validada: pontuação por sinais contábeis, âncoras no Item 8, no parecer do
auditor e no balanço, busca em anexos e na submissão completa.

In [ ]:
IGNORAR = re.compile(r"(?i)^(r\d+\.htm|report\d*\.htm|.*-index.*\.html?|"
                     r"\d{10}-\d{2}-\d{6}\.txt|.*_?cal\.|.*_?def\.|.*_?lab\.|.*_?pre\.|"
                     r".*\.xml|.*\.xsd|.*\.jpg|.*\.png|.*\.gif)$")
EXTENSOES = (".htm", ".html", ".txt", ".pdf")

def arquivos_do_protocolo(url_pasta, excluir=()):
    """Documentos do protocolo, do maior para o menor, incluindo os anexos em PDF."""
    txt = baixar(urljoin(url_pasta, "index.json"))
    if not txt:
        return []
    try:
        itens = json.loads(txt)["directory"]["item"]
    except (ValueError, KeyError):
        return []
    saida = []
    for i in itens:
        nome = i.get("name", "")
        if not nome.lower().endswith(EXTENSOES):
            continue
        if IGNORAR.match(nome) or nome in excluir:
            continue
        saida.append((nome, int(i.get("size") or 0)))
    saida.sort(key=lambda x: x[1], reverse=True)
    return [n for n, _ in saida]

P_DOCUMENTO = re.compile(r"(?is)<DOCUMENT>.*?<TYPE>([^\r\n<]*).*?<FILENAME>([^\r\n<]*)"
                         r".*?<TEXT>(.*?)</TEXT>")

def documentos_da_submissao(url_submissao):
    """Último recurso: a submissão completa reúne todos os documentos do protocolo."""
    bruto = baixar(url_submissao)
    if not bruto:
        return []
    saida = []
    for tipo, nome, corpo in P_DOCUMENTO.findall(bruto):
        nome = nome.strip()
        if not nome.lower().endswith((".htm", ".html", ".txt")):
            continue
        if len(corpo) > 40000:
            saida.append((nome, corpo))
    saida.sort(key=lambda x: len(x[1]), reverse=True)
    return saida[:6]

def documento_principal(linha):
    """O documento principal vem da tabela do painel; a listagem da pasta é a alternativa."""
    doc = linha.get("documento_principal")
    if not (isinstance(doc, str) and doc):
        nomes = arquivos_do_protocolo(linha["url_pasta"])
        doc = nomes[0] if nomes else None
    sic = str(linha.get("sic") or "")
    return (linha["url_pasta"] + doc if doc else None), doc, sic

In [ ]:
ENTIDADES = re.compile(r"&nbsp;|&#160;|&#xa0;|&#xA0;")
TAGS      = re.compile(r"(?is)<(script|style)\b.*?</\1>|<[^>]+>")

def texto_e_mapa(html):
    """Texto visível em minúsculas, espaços normalizados, com mapa texto -> posição no HTML."""
    saida, mapa, ultimo, espaco = [], [], 0, True
    def empurrar(ch, pos):
        nonlocal espaco
        if ch.isspace():
            if espaco:
                return
            saida.append(" "); mapa.append(pos); espaco = True
        else:
            saida.append(ch.lower()); mapa.append(pos); espaco = False
    for m in TAGS.finditer(html):
        for k, ch in enumerate(html[ultimo:m.start()]):
            empurrar(ch, ultimo + k)
        empurrar(" ", m.start())
        ultimo = m.end()
    for k, ch in enumerate(html[ultimo:]):
        empurrar(ch, ultimo + k)
    return "".join(saida), mapa

# Títulos de início e de fim
P_10K_INI = re.compile(r"item\s*8[\.\:\)\-—\s]{0,6}financial\s+statements")
P_10K_FIM = re.compile(r"item\s*9[a-c]?[\.\:\)\-—\s]{0,6}(changes\s+in\s+and|controls\s+and\s+procedures|other\s+information)")
P_20F_INI = re.compile(r"item\s*18[\.\:\)\-—\s]{0,6}financial\s+statements")
P_20F_ALT = re.compile(r"item\s*17[\.\:\)\-—\s]{0,6}financial\s+statements")
P_20F_FIM = re.compile(r"item\s*19[\.\:\)\-—\s]{0,6}exhibit")
P_AUDITOR = re.compile(r"report\s+of\s+independent|independent\s+auditor'?s?\s+report|"
                       r"auditors?'?\s+report\s+to|report\s+of\s+the\s+independent")
P_ASSINAT = re.compile(r"\bsignatures?\b")

# ---------------------------------------------------------------------------
# Sinais que caracterizam um conjunto de demonstrações contábeis
# ---------------------------------------------------------------------------
SINAIS = {
    "balanco":    re.compile(r"balance\s+sheets?|statements?\s+of\s+financial\s+position"),
    "notas":      re.compile(r"notes\s+to\s+.{0,60}?financial\s+statements|"
                             r"notes\s+to\s+the\s+accounts"),
    "resultado":  re.compile(r"statements?\s+of\s+((net|total|consolidated|combined)\s+){0,2}"
                             r"(operations|income|comprehensive|profit|earnings|loss)|"
                             r"income\s+statements?"),
    "fluxo":      re.compile(r"statements?\s+of\s+cash\s+flows?|cash\s+flow\s+statements?"),
    "patrimonio": re.compile(r"statements?\s+of\s+(changes\s+in\s+)?"
                             r"(shareholders|stockholders|owners|equity)"),
    "auditor":    P_AUDITOR,
}
OBRIGATORIOS = ("balanco", "notas")     # sem estes dois, o trecho é texto narrativo, não demonstração
PONTOS_MINIMOS = 4                      # de 6 sinais
PONTOS_BONS    = 5                      # a partir daqui, não vale procurar em outros documentos

# Conteúdo numérico: demonstrações contábeis são densas em números; índices e sumários não.
P_NUMERO = re.compile(r"\d{1,3}(?:,\d{3})+|\d+\.\d{2}\b")
MIN_NUMEROS = 150          # abaixo disso o trecho é índice ou remissão, não demonstração
NUMEROS_BONS = 400         # a partir daqui, não vale procurar em outros documentos
RETENCAO_MINIMA = 0.6      # trecho mais curto que preserve ao menos esta fração dos números

# Declarações de ausência (fundos de securitização, empresas-veículo)
P_OMITIDO = re.compile(r"^.{0,400}?(omitted|not\s+applicable|none\.)", re.S)

MIN_ACEITAVEL = 4000

def pontuar(trecho):
    """(sinais contábeis presentes, quantidade de números). Zero sinais se faltar um obrigatório."""
    presentes = {k for k, p in SINAIS.items() if p.search(trecho)}
    numeros = len(P_NUMERO.findall(trecho))
    if any(o not in presentes for o in OBRIGATORIOS):
        return 0, numeros
    return len(presentes), numeros

def melhor_par(texto, p_ini, p_fim):
    """Par (início, fim) mais distante — evita o sumário, onde os títulos ficam colados."""
    inis = [m.start() for m in p_ini.finditer(texto)]
    fins = [m.start() for m in p_fim.finditer(texto)]
    melhor, tamanho = None, 0
    for i in inis:
        seguintes = [f for f in fins if f > i]
        f = seguintes[0] if seguintes else len(texto)
        if f - i > tamanho:
            melhor, tamanho = (i, f), f - i
    return melhor

def candidatos(texto, formulario):
    """Trechos a testar: pelo item, pelo parecer do auditor e, quando cabe, o documento inteiro."""
    saida = []
    if formulario == "10-K":
        p = melhor_par(texto, P_10K_INI, P_10K_FIM)
        if p:
            saida.append((p[0], p[1], "item8"))
    elif formulario == "20-F":
        for padrao, nome in ((P_20F_INI, "item18"), (P_20F_ALT, "item17")):
            p = melhor_par(texto, padrao, P_20F_FIM)
            if p:
                saida.append((p[0], p[1], nome))
    # Âncoras: o parecer do auditor e o título do balanço. As duas são necessárias porque a
    # ordem varia — em boa parte dos arquivamentos europeus o parecer vem depois das
    # demonstrações, e ancorar só nele deixaria as demonstrações de fora do trecho.
    for padrao, nome in ((P_AUDITOR, "paginas_F"), (SINAIS["balanco"], "balanco")):
        for m in list(padrao.finditer(texto))[:5]:
            ini = max(0, m.start() - 60)       # recua o bastante para não cortar o título ao meio
            fins = [s.start() for s in P_ASSINAT.finditer(texto) if s.start() > ini + MIN_ACEITAVEL]
            saida.append((ini, fins[-1] if fins else len(texto), nome))
    if formulario in ("40-F", "anexo"):
        saida.append((0, len(texto), "documento_integral"))
    return saida

def recortar(html, formulario):
    """Melhor trecho do documento: (html_da_secao, metodo, n_caracteres, pontos, numeros).
    metodo 'sem_demonstracoes' quando o item existe e está declarado como omitido."""
    html = ENTIDADES.sub(" ", html)
    texto, mapa = texto_e_mapa(html)
    if not texto:
        return None, "sem_texto", 0, 0, 0

    validos, omitido = [], False
    for ini, fim, metodo in candidatos(texto, formulario):
        trecho = texto[ini:fim]
        if len(trecho) < MIN_ACEITAVEL:
            if P_OMITIDO.search(trecho):
                omitido = True
            continue
        pontos, numeros = pontuar(trecho)
        if pontos < PONTOS_MINIMOS or numeros < MIN_NUMEROS:
            continue
        validos.append((pontos, numeros, ini, fim, metodo, len(trecho)))

    if not validos:
        return None, ("sem_demonstracoes" if omitido else "nao_localizado"), 0, 0, 0

    # Escolha em duas etapas. Primeiro o maior número de sinais e o maior conteúdo numérico,
    # que é o que separa as demonstrações do índice que as lista. Depois, entre os trechos que
    # preservam a maior parte desse conteúdo, o mais curto — assim o resultado fica nas
    # demonstrações em vez de abarcar o relatório inteiro.
    melhor_pontos = max(v[0] for v in validos)
    fortes = [v for v in validos if v[0] == melhor_pontos]
    teto = max(v[1] for v in fortes)
    proximos = [v for v in fortes if v[1] >= RETENCAO_MINIMA * teto]
    pontos, numeros, ini, fim, metodo, n = min(proximos, key=lambda v: v[5])
    ini_html = mapa[ini]
    fim_html = mapa[fim - 1] if fim - 1 < len(mapa) else len(html)
    return html[ini_html:fim_html], metodo, n, pontos, numeros

MOLDE = """<html><head><meta charset="utf-8"><style>
 @page {{ size: A4; margin: 12mm }}
 body {{ font-family: Georgia, serif; font-size: 9pt; line-height: 1.35 }}
 table {{ border-collapse: collapse; font-size: 7.5pt; width: 100% }}
 td, th {{ padding: 1px 3px; vertical-align: bottom }}
 img {{ display: none }}
 .cabecalho {{ font-size: 8pt; color: #555; border-bottom: 1px solid #999;
               margin-bottom: 8pt; padding-bottom: 4pt }}
</style></head><body>
<div class="cabecalho">{titulo}</div>
{corpo}
</body></html>"""

def texto_de_pdf(bytes_pdf):
    """Texto de um anexo já entregue em PDF, para que ele possa ser avaliado como os demais."""
    with tempfile.NamedTemporaryFile("wb", suffix=".pdf", delete=False) as f:
        f.write(bytes_pdf); tmp = f.name
    try:
        r = subprocess.run(["pdftotext", "-q", tmp, "-"], capture_output=True, text=True,
                           timeout=300)
        return r.stdout
    except Exception:
        return ""
    finally:
        os.unlink(tmp)

def avaliar_pdf(bytes_pdf):
    """(pontos, numeros) de um anexo em PDF."""
    txt = re.sub(r"\s+", " ", texto_de_pdf(bytes_pdf)).lower()
    if len(txt) < MIN_ACEITAVEL:
        return 0, 0
    return pontuar(txt)

def gerar_pdf(html_secao, titulo, destino):
    with tempfile.NamedTemporaryFile("w", suffix=".html", delete=False, encoding="utf-8") as f:
        f.write(MOLDE.format(titulo=titulo, corpo=html_secao)); tmp = f.name
    try:
        subprocess.run(["wkhtmltopdf", "--quiet", "--enable-local-file-access", "--no-images",
                        "--load-error-handling", "ignore", "--load-media-error-handling", "ignore",
                        "--disable-external-links", "--footer-right", "[page]/[topage]",
                        "--footer-font-size", "7", tmp, destino],
                       capture_output=True, timeout=600)
        return os.path.exists(destino) and os.path.getsize(destino) > 1000
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(tmp)

In [ ]:
LOG    = os.path.join(PASTA, "log", "extracao.csv")
FALHAS = os.path.join(PASTA, "log", "falhas.csv")
COLUNAS = ("accession,cik,empresa,ticker,formulario,data,sic,metodo,documento,pontos,numeros,"
           "caracteres,kb_pdf,url")
COL_FALHA = "accession,cik,empresa,formulario,data,sic,motivo,url,documentos_examinados"

_log_lock = threading.Lock()

def registrar(caminho, linha, cabecalho):
    with _log_lock:
        novo = not os.path.exists(caminho)
        with open(caminho, "a", encoding="utf-8") as f:
            if novo:
                f.write(cabecalho + "\n")
            f.write(linha + "\n")

def concluidos():
    feitos = set()
    for c in (LOG, FALHAS):
        if os.path.exists(c):
            feitos |= set(pd.read_csv(c)["accession"].astype(str))
    return feitos

def limpo(s):
    return str(s).replace(",", " ").replace("\n", " ")[:80]

def extrair(linha):
    """Percorre os documentos do protocolo e devolve o trecho de melhor conteúdo contábil.
    (conteudo, metodo, n, pontos, numeros, url, documento, sic, examinados)"""
    url, nome, sic = documento_principal(linha)
    melhor, omitido, examinados = None, False, []

    def considerar(bruto, u, doc, forma):
        """Avalia um documento. bruto em texto (HTML) ou bytes (PDF já pronto)."""
        nonlocal melhor, omitido
        examinados.append(doc)
        if isinstance(bruto, bytes):                     # anexo entregue em PDF
            pontos, numeros = avaliar_pdf(bruto)
            if pontos >= PONTOS_MINIMOS and numeros >= MIN_NUMEROS:
                chave = (pontos, numeros)
                if melhor is None or chave > (melhor[3], melhor[4]):
                    melhor = (bruto, "pdf_original", 0, pontos, numeros, u, doc)
                return pontos, numeros
            return 0, 0
        secao, metodo, n, pontos, numeros = recortar(bruto, forma)
        if metodo == "sem_demonstracoes":
            omitido = True
        if secao:
            chave = (pontos, numeros)
            if melhor is None or chave > (melhor[3], melhor[4]):
                melhor = (secao, metodo, n, pontos, numeros, u, doc)
            return pontos, numeros
        return 0, 0

    def bom(par):
        return par[0] >= PONTOS_BONS and par[1] >= NUMEROS_BONS

    if linha["formulario"] != "40-F" and url:
        bruto = baixar(url)
        if bruto and bom(considerar(bruto, url, nome, linha["formulario"])):
            return melhor + (sic, examinados)
        if omitido:                       # item existe e está declarado como omitido
            return (None, "sem_demonstracoes", 0, 0, 0, url, nome, sic, examinados)

    # Demais documentos do protocolo: anexos do 40-F, demonstrações em arquivo separado
    for outro in arquivos_do_protocolo(linha["url_pasta"], excluir=(nome,) if nome else ())[:12]:
        u = linha["url_pasta"] + outro
        binario = outro.lower().endswith(".pdf")
        bruto = baixar(u, binario=binario)
        if not bruto:
            continue
        if bom(considerar(bruto, u, outro, "anexo")):
            break

    # Último recurso: a submissão completa, que traz documentos fora da listagem da pasta
    if melhor is None and not omitido:
        for doc, corpo in documentos_da_submissao(linha["url_submissao_completa"]):
            if bom(considerar(corpo, linha["url_submissao_completa"], doc, "anexo")):
                break

    if melhor:
        conteudo, metodo, n, pontos, numeros, u, doc = melhor
        if doc != nome:
            metodo += "_em_anexo"
        return (conteudo, metodo, n, pontos, numeros, u, doc, sic, examinados)
    return (None, "sem_demonstracoes" if omitido else "nao_localizado",
            0, 0, 0, url, nome, sic, examinados)

def processar(linha):
    acc = linha["accession"]
    try:
        secao, metodo, n, pontos, numeros, url, doc, sic, examinados = extrair(linha)
        base_falha = (f"{acc},{linha['cik']},{limpo(linha['empresa'])},{linha['formulario']},"
                      f"{linha['data']},{sic}")
        if not secao:
            registrar(FALHAS, f"{base_falha},{metodo},{url},{' '.join(examinados[:10])}",
                      COL_FALHA)
            return metodo if metodo == "sem_demonstracoes" else "falha"

        pasta_ano = os.path.join(PASTA, "pdf", str(linha["ano"]), linha["formulario"])
        os.makedirs(pasta_ano, exist_ok=True)
        tic = linha.get("ticker") if pd.notna(linha.get("ticker")) else "NA"
        base = f"{linha['cik']}_{tic}_{linha['ano']}_{acc}"
        destino = os.path.join(pasta_ano, base + ".pdf")

        titulo = (f"{limpo(linha['empresa'])} — CIK {linha['cik']} — {linha['formulario']} — "
                  f"protocolo {linha['data']} — accession {acc}")
        if isinstance(secao, bytes):          # anexo já entregue em PDF pela própria empresa
            with open(destino, "wb") as f:
                f.write(secao)
        elif not gerar_pdf(secao, titulo, destino):
            registrar(FALHAS, f"{base_falha},pdf_falhou,{url},", COL_FALHA)
            return "falha"

        if SALVAR_HTML_DA_SECAO and not isinstance(secao, bytes):
            ph = os.path.join(PASTA, "html_secao", str(linha["ano"]))
            os.makedirs(ph, exist_ok=True)
            with open(os.path.join(ph, base + ".html"), "w", encoding="utf-8") as f:
                f.write(secao)

        kb = os.path.getsize(destino) // 1024
        registrar(LOG, f"{acc},{linha['cik']},{limpo(linha['empresa'])},{tic},"
                       f"{linha['formulario']},{linha['data']},{sic},{metodo},{doc},{pontos},"
                       f"{numeros},{n},{kb},{url}", COLUNAS)
        return "ok"
    except Exception as e:
        registrar(FALHAS, f"{acc},{linha['cik']},{limpo(linha.get('empresa'))},"
                          f"{linha['formulario']},{linha['data']},,{type(e).__name__},,",
                  COL_FALHA)
        return "erro"

## 3. Leitura do painel

O arquivo da Compustat traz uma linha por empresa-ano. Interessam quatro colunas: `cik`,
`datadate` (fim do exercício), `conm` (nome) e `Year`.

In [ ]:
painel = pd.read_csv(ARQUIVO_PAINEL)
painel["cik"] = pd.to_numeric(painel["cik"], errors="coerce").astype("Int64")
painel["datadate"] = pd.to_datetime(painel["datadate"])
painel = painel.dropna(subset=["cik"]).drop_duplicates(["cik", "datadate"])

print("observações:", len(painel), "| empresas:", painel["cik"].nunique())
print(painel.groupby("Year").size().to_string())

observações: 66680 | empresas: 10163
Year
2017    7357
2018    7271
2019    7338
2020    7429
2021    7597
2022    7611
2023    7475
2024    7213
2025    6749
2026     640


## 4. Tabela de protocolos

Uma consulta por empresa à base de submissões da SEC. Para cada uma, guardamos todos os
relatórios anuais com a data de encerramento do período, o número do protocolo e o nome do
documento principal. O resultado é gravado em `indice/protocolos_painel.csv`; nas execuções
seguintes ele é lido do disco.

In [ ]:
CAMINHO_PROTOCOLOS = os.path.join(PASTA, "indice", "protocolos_painel.csv")

def protocolos_da_empresa(cik):
    """Relatórios anuais de uma empresa: forma, período, protocolo e documento principal."""
    linhas = []
    txt = baixar(f"https://data.sec.gov/submissions/CIK{int(cik):010d}.json")
    if not txt:
        return linhas
    try:
        d = json.loads(txt)
    except ValueError:
        return linhas
    sic = str(d.get("sic", ""))
    blocos = [d.get("filings", {}).get("recent", {})]
    for extra in d.get("filings", {}).get("files", []):
        t2 = baixar("https://data.sec.gov/submissions/" + extra["name"])
        if t2:
            try:
                blocos.append(json.loads(t2))
            except ValueError:
                pass
    for b in blocos:
        campos = ["accessionNumber", "form", "filingDate", "reportDate", "primaryDocument"]
        if not all(c in b for c in campos):
            continue
        for acc, forma, data, periodo, doc in zip(*[b[c] for c in campos]):
            if forma in FORMULARIOS:
                linhas.append((int(cik), sic, forma, acc, data, periodo, doc))
    return linhas

if os.path.exists(CAMINHO_PROTOCOLOS):
    protocolos = pd.read_csv(CAMINHO_PROTOCOLOS)
    print("tabela lida do disco:", len(protocolos), "protocolos")
else:
    ciks = sorted(painel["cik"].dropna().unique())
    print("empresas a consultar:", len(ciks), "— cerca de", round(len(ciks)/REQ_POR_SEGUNDO/60), "min")
    todas = []
    with ThreadPoolExecutor(max_workers=N_THREADS) as pool:
        for i, linhas in enumerate(pool.map(protocolos_da_empresa, ciks), 1):
            todas.extend(linhas)
            if i % 500 == 0:
                print(f"{i}/{len(ciks)} empresas | {len(todas)} protocolos")
    protocolos = pd.DataFrame(todas, columns=["cik", "sic", "formulario", "accession",
                                              "data_protocolo", "periodo", "documento_principal"])
    protocolos.to_csv(CAMINHO_PROTOCOLOS, index=False)
    print("gravado em", CAMINHO_PROTOCOLOS)

print(protocolos["formulario"].value_counts().to_string())

tabela lida do disco: 123896 protocolos
formulario
10-K    106487
20-F     14700
40-F      2709


## 5. Casamento entre painel e protocolos

Primeiro pela data de encerramento do período informada pela SEC, que é o critério correto.
Para as observações que sobrarem, uma segunda tentativa pela data de protocolo, aceitando um
relatório arquivado entre 15 e 400 dias após o fim do exercício.

In [ ]:
p = protocolos.copy()
p["cik"] = pd.to_numeric(p["cik"], errors="coerce").astype("int64")
p["periodo"] = pd.to_datetime(p["periodo"], errors="coerce")
p["data_protocolo"] = pd.to_datetime(p["data_protocolo"], errors="coerce")
p = p.dropna(subset=["periodo"]).sort_values("periodo")

alvo = painel[["cik", "datadate", "conm", "tic", "Year"]].copy()
alvo["cik"] = alvo["cik"].astype("int64")
alvo = alvo.sort_values("datadate")

# 1) pelo período do relatório
casado = pd.merge_asof(alvo, p, left_on="datadate", right_on="periodo", by="cik",
                       direction="nearest", tolerance=pd.Timedelta(days=TOLERANCIA_DIAS))

# 2) o que sobrou, pela data de protocolo
falta = casado["accession"].isna()
if falta.any():
    resto = alvo[alvo.set_index(["cik", "datadate"]).index.isin(
        casado.loc[falta].set_index(["cik", "datadate"]).index)]
    p2 = p.sort_values("data_protocolo")
    segunda = pd.merge_asof(resto.sort_values("datadate"), p2,
                            left_on="datadate", right_on="data_protocolo", by="cik",
                            direction="forward", tolerance=pd.Timedelta(days=400))
    # mantém também as que não casaram, para ficarem registradas no relatório
    casado = pd.concat([casado[~falta], segunda], ignore_index=True)

casado["encontrado"] = casado["accession"].notna()
print("empresas-ano casadas: %d de %d (%.1f%%)"
      % (casado["encontrado"].sum(), len(alvo), casado["encontrado"].mean() * 100))
print(casado[casado["encontrado"]]["formulario"].value_counts().to_string())
print(casado.groupby("Year")["encontrado"].mean().round(3).to_string())

casado.to_csv(os.path.join(PASTA, "indice", "painel_casado.csv"), index=False)

empresas-ano casadas: 52233 de 66680 (78.3%)
formulario
10-K    43754
20-F     7256
40-F     1223
Year
2017    0.750
2018    0.752
2019    0.741
2020    0.759
2021    0.802
2022    0.800
2023    0.792
2024    0.811
2025    0.842
2026    0.820


In [ ]:
# Lista de trabalho: um registro por protocolo, no formato que o extrator espera
t = casado[casado["encontrado"]].copy()
t["accession_sem_hifen"] = t["accession"].str.replace("-", "", regex=False)
t["url_pasta"] = ("https://www.sec.gov/Archives/edgar/data/" + t["cik"].astype(str)
                  + "/" + t["accession_sem_hifen"] + "/")
t["url_submissao_completa"] = (t["url_pasta"] + t["accession"] + ".txt")
t = t.rename(columns={"conm": "empresa", "tic": "ticker", "Year": "ano"})
t["data"] = t["data_protocolo"].dt.strftime("%Y-%m-%d")

trabalho = t.drop_duplicates("accession")[
    ["cik", "empresa", "ticker", "formulario", "data", "ano", "sic",
     "accession", "documento_principal", "url_pasta", "url_submissao_completa"]]

print("protocolos distintos a extrair:", len(trabalho))
print(trabalho.groupby(["ano", "formulario"]).size().unstack(fill_value=0).to_string())
trabalho.to_csv(os.path.join(PASTA, "indice", "trabalho.csv"), index=False)

protocolos distintos a extrair: 51491
formulario  10-K  20-F  40-F
ano                         
2017        4714   597   122
2018        4639   627   117
2019        4575   667   114
2020        4708   718   131
2021        5037   816   154
2022        4999   867   147
2023        4806   896   138
2024        4666   972   134
2025        4461   998   146
2026         421    93    11


## 6. Aproveitamento do que já foi extraído

Os PDFs de 2017 em diante que já estão na pasta `SEC_EDGAR` fazem parte deste painel. As duas
células abaixo movem esses arquivos para a estrutura nova, renomeando-os pelo ano fiscal, e
registram no log da pasta nova para que não sejam baixados outra vez. Mover arquivos dentro do
Drive é instantâneo — não há novo download.

Rode uma vez só. Se você não tem a pasta antiga, pule esta seção.

In [ ]:
import os, glob, shutil, pandas as pd

ORIGEM = "/content/drive/MyDrive/SEC_EDGAR"

destino_de = {}
for r in trabalho.to_dict("records"):
    tic = r["ticker"] if pd.notna(r["ticker"]) else "NA"
    nome = "%s_%s_%s_%s.pdf" % (r["cik"], tic, r["ano"], r["accession"])
    destino_de[r["accession"]] = os.path.join(
        PASTA, "pdf", str(r["ano"]), r["formulario"], nome)

antigos = glob.glob(os.path.join(ORIGEM, "pdf", "**", "*.pdf"), recursive=True)
print("PDFs na pasta antiga:", len(antigos))

movidos, fora = 0, 0
for caminho in antigos:
    acc = os.path.basename(caminho).rsplit(".", 1)[0].split("_")[-1]
    novo = destino_de.get(acc)
    if novo is None:
        fora += 1
        continue
    if os.path.exists(novo):
        continue
    os.makedirs(os.path.dirname(novo), exist_ok=True)
    shutil.move(caminho, novo)
    movidos += 1

print("aproveitados:", movidos)
print("fora do painel (ficaram onde estavam):", fora)

In [2]:
antigo_log = pd.read_csv(os.path.join(ORIGEM, "log", "extracao.csv"))
uteis = antigo_log[antigo_log["accession"].isin(trabalho["accession"])]
print("linhas de log aproveitadas:", len(uteis))

if os.path.exists(LOG):
    atual = pd.read_csv(LOG)
    juntos = pd.concat([atual, uteis]).drop_duplicates("accession")
else:
    juntos = uteis
juntos.to_csv(LOG, index=False)
print("log da pasta nova:", len(juntos), "registros")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/SEC_EDGAR/log/extracao.csv'

## 7. Extração

In [ ]:
from time import time as _t

ANOS_DESTA_RODADA = sorted(trabalho["ano"].unique())     # ex.: [2017, 2018]

feitos = concluidos()
lote = trabalho[trabalho["ano"].isin(ANOS_DESTA_RODADA)]
lote = lote[~lote["accession"].isin(feitos)]
registros = lote.to_dict("records")

print("a processar:", len(registros))
print("tempo estimado: %.1f h" % (len(registros) / 2000))

t0 = _t()
cont = {}
with ThreadPoolExecutor(max_workers=N_THREADS) as pool:
    for i, r in enumerate(pool.map(processar, registros), 1):
        cont[r] = cont.get(r, 0) + 1
        if i % 200 == 0 or i == len(registros):
            falta = (_t() - t0) / i * (len(registros) - i) / 3600
            print(i, "/", len(registros), cont, "faltam %.1f h" % falta)

print("concluído:", cont)

a processar: 51491
tempo estimado: 25.7 h
200 / 51491 {'ok': 199, 'sem_demonstracoes': 1} faltam 12.3 h
400 / 51491 {'ok': 394, 'sem_demonstracoes': 3, 'falha': 3} faltam 12.9 h
600 / 51491 {'ok': 593, 'sem_demonstracoes': 3, 'falha': 4} faltam 12.6 h
800 / 51491 {'ok': 789, 'sem_demonstracoes': 3, 'falha': 8} faltam 12.2 h
1000 / 51491 {'ok': 985, 'sem_demonstracoes': 3, 'falha': 12} faltam 12.6 h
1200 / 51491 {'ok': 1182, 'sem_demonstracoes': 3, 'falha': 15} faltam 13.4 h
1400 / 51491 {'ok': 1381, 'sem_demonstracoes': 4, 'falha': 15} faltam 13.8 h
1600 / 51491 {'ok': 1579, 'sem_demonstracoes': 5, 'falha': 16} faltam 14.2 h
1800 / 51491 {'ok': 1779, 'sem_demonstracoes': 5, 'falha': 16} faltam 14.5 h
2000 / 51491 {'ok': 1978, 'sem_demonstracoes': 5, 'falha': 17} faltam 14.6 h
2200 / 51491 {'ok': 2175, 'sem_demonstracoes': 5, 'falha': 20} faltam 14.8 h
2400 / 51491 {'ok': 2375, 'sem_demonstracoes': 5, 'falha': 20} faltam 14.8 h
2600 / 51491 {'ok': 2574, 'sem_demonstracoes': 5, 'falha': 

## 8. Situação e conferência

A primeira célula responde onde estamos; a segunda verifica o conteúdo de uma amostra dos PDFs.

In [ ]:
import glob

log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame(columns=["accession"])
fal = pd.read_csv(FALHAS) if os.path.exists(FALHAS) else pd.DataFrame(columns=["accession"])
tratados = set(log["accession"].astype(str)) | set(fal["accession"].astype(str))

situacao = pd.DataFrame({
    "a_extrair": trabalho.groupby("ano").size(),
    "extraidos": log.assign(ano=log["data"].str[:4].astype(int)).groupby("ano").size()
                 if len(log) else 0,
})
situacao["faltam"] = (trabalho[~trabalho["accession"].isin(tratados)]
                      .groupby("ano").size().reindex(situacao.index).fillna(0).astype(int))
print(situacao.fillna(0).astype(int).to_string())

pdfs = glob.glob(os.path.join(PASTA, "pdf", "**", "*.pdf"), recursive=True)
print("PDFs:", len(pdfs))
print("espaço: %.2f GB" % (sum(os.path.getsize(p) for p in pdfs) / 1e9))

In [ ]:
import random

P_CHECAGEM = {
    "balanco": r"balance sheets?|statements? of financial position",
    "resultado": r"statements? of ((net|total|consolidated) ){0,2}(operations|income|comprehensive|profit|earnings|loss)",
    "fluxo": r"statements? of cash flows?|cash flow statements?",
    "notas": r"notes to .{0,60}?financial statements|notes to the accounts",
}

amostra = random.sample(pdfs, min(20, len(pdfs)))
linhas = []
for p in amostra:
    bruto = subprocess.run(["pdftotext", "-q", p, "-"], capture_output=True,
                           text=True, timeout=180).stdout
    paginas = bruto.count("\f") or 1
    txt = re.sub(r"\s+", " ", bruto).lower()
    item = {"arquivo": os.path.basename(p), "paginas": paginas}
    item.update({k: bool(re.search(v, txt)) for k, v in P_CHECAGEM.items()})
    linhas.append(item)

conf = pd.DataFrame(linhas)
conf["completo"] = conf[list(P_CHECAGEM)].all(axis=1)
print("completos:", int(conf["completo"].sum()), "de", len(conf))
print(conf[~conf["completo"]].to_string(index=False) if (~conf["completo"]).any() else "sem pendências")

## 9. Compactar para download

Volumes de 1,5 GB por ano fiscal, gravados em `zip/`.

In [ ]:
import zipfile, glob

LIMITE_GB = 1.5
destino = os.path.join(PASTA, "zip")
os.makedirs(destino, exist_ok=True)
raiz = os.path.join(PASTA, "pdf")

for ano in sorted(d for d in os.listdir(raiz) if d.isdigit()):
    arquivos = sorted(glob.glob(os.path.join(raiz, ano, "**", "*.pdf"), recursive=True))
    if not arquivos:
        continue
    limite = LIMITE_GB * 1e9
    volumes, atual, tam = [], [], 0
    for a in arquivos:
        t = os.path.getsize(a)
        if atual and tam + t > limite:
            volumes.append(atual)
            atual, tam = [], 0
        atual.append(a)
        tam += t
    volumes.append(atual)
    print(ano, len(arquivos), "PDFs em", len(volumes), "volume(s)")
    for i, volume in enumerate(volumes, 1):
        nome = os.path.join(destino, "SEC_%s_parte%02d.zip" % (ano, i))
        with zipfile.ZipFile(nome, "w", zipfile.ZIP_STORED) as z:
            for a in volume:
                z.write(a, os.path.relpath(a, raiz))
        print("  ", os.path.basename(nome), "%.2f GB" % (os.path.getsize(nome) / 1e9))

## Notas

- **Nome dos arquivos.** `CIK_ticker_anofiscal_accession.pdf`. O `accession` liga cada PDF à
  linha do painel em `indice/painel_casado.csv`.
- **Empresas sem correspondência.** Ficam registradas em `painel_casado.csv` com `encontrado`
  igual a falso. São, em boa parte, empresas canadenses que arquivam apenas no SEDAR e nunca
  protocolaram na SEC.
- **Retomada.** Depois de qualquer queda de sessão: seções 1 e 2, depois a leitura do painel, a
  tabela de protocolos (que agora vem do disco), o casamento e a extração.
- **Aproveitamento do trabalho anterior.** Os PDFs de 2017 em diante que já estão na pasta
  `SEC_EDGAR` cobrem parte deste painel. Para não baixá-los de novo, copie o `log/extracao.csv`
  antigo para a pasta nova antes da primeira execução; o programa passa a considerá-los
  concluídos. Se preferir recomeçar limpo, ignore este item.